# Supervised Learning and K Nearest Neighbors Exercises

![knn.png](Assets/knn.png)

# Learning Objectives:

- Explicarea învățării supravegheate și modul în care aceasta poate fi aplicată la problemele de regresie și clasificare 
- Aplicarea algoritmului K-Nearest Neighbor (KNN) pentru clasificare
- Aplicați Intel® Extension for Scikit-learn* pentru a valorifica capacitățile de calcul subiacente ale hardware-ului



## Introduction
Pentru exercițiile din prima săptămână, vom utiliza date privind pierderea clienților din industria telecomunicațiilor. Fișierul de date se numește  `Orange_Telecom_Churn_Data.csv`. Vom încărca aceste date împreună, vom efectua o preprocesare și vom utiliza vecinii K-nearest pentru a prezice pierderea clienților pe baza caracteristicilor contului.

In [ ]:
from __future__ import print_function
import os
data_path = ['data']

from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

from sklearnex import patch_sklearn
patch_sklearn()



## Question 1
* Începeți prin a importa datele. Examinați coloanele și datele.
* Observați că datele conțin un stat, un cod regional și un număr de telefon. Credeți că acestea sunt caracteristici bune de utilizat la construirea unui model de învățare automată? De ce sau de ce nu? 

Nu le vom utiliza, așa că pot fi eliminate din date.

In [ ]:
import pandas as pd

# Import the data using the file path
filepath = os.sep.join(data_path + ['Orange_Telecom_Churn_Data.csv'])
data = pd.read_csv(filepath)

In [ ]:
data.head(1).T

In [ ]:
# Remove extraneous columns
data.drop(['state', 'area_code', 'phone_number'], axis=1, inplace=True)

In [ ]:
data.columns

## Question 2

* Observați că unele dintre coloane sunt date categorice, iar altele sunt date flotante. Aceste caracteristici vor trebui să fie codificate numeric folosind una dintre metodele din curs.
* În cele din urmă, rețineți din prelegere că metoda K-nearest neighbors necesită date scalate. Scalați datele utilizând una dintre metodele de scalare discutate în prelegere.

In [ ]:
lb = LabelBinarizer()

for col in ['intl_plan', 'voice_mail_plan', 'churned']:
    data[col] = lb.fit_transform(data[col])
print(data)

In [ ]:
# Mute the sklearn warning
import warnings
warnings.filterwarnings('ignore', module='sklearn')

msc = MinMaxScaler()

data = pd.DataFrame(msc.fit_transform(data),  # this is an np.array, not a dataframe.
                    columns=data.columns)

## Question 3

Separați coloanele de caracteristici (totul, cu excepția churned) de etichetă (churned). Acest lucru va crea două tabele.
Adaptați un model K-nearest neighbors cu o valoare 'k=3' la aceste date și preziceți rezultatul pe aceleași date.

In [ ]:
# Get a list of all the columns that don't contain the label
x_cols = [x for x in data.columns if x != 'churned']

# Split the data into two dataframes
X_data = data[x_cols]
y_data = data['churned']

# # alternatively:
# X_data = data.copy()
# y_data = X_data.pop('churned')

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)

knn = knn.fit(X_data, y_data)

y_pred = knn.predict(X_data)

## Question 4

Metodele de măsurare a erorii nu au fost discutate încă în clasă, dar precizia este ușor de înțeles--este pur și simplu procentul de etichete care au fost prezise corect (fie adevărat, fie fals). 

* Scrieți o funcție pentru a calcula acuratețea folosind etichetele reale și prezise.
* Folosind funcția, calculați precizia acestui model K-nearest neighbors pe date.

In [ ]:
# Function to calculate the % of values that were correctly predicted

def accuracy(real, predict):
    return sum(y_data == y_pred) / float(real.shape[0])

In [ ]:
print(accuracy(y_data, y_pred))

## Question 5
* Adaptați din nou modelul K-nearest neighbors cu `n_neighbors=3`, dar de data aceasta utilizați distanța pentru ponderi. Calculați precizia folosind funcția pe care ați creat-o mai sus. 
* Adaptați un alt model K-nearest neighbors. De data aceasta utilizați ponderi uniforme, dar setați parametrul de putere pentru metrica distanței Minkowski la 1 (`p=1`), adică distanța Manhattan.

Atunci când distanțele ponderate sunt utilizate pentru partea 1 a acestei întrebări, pentru acuratețe ar trebui returnată o valoare de 1,0. De ce credeți că este așa? 
* Hint:* facem predicții pe baza datelor, iar cu KNN modelul *este* datele. Vom învăța cum să evităm această capcană în cursul următor.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')

knn = knn.fit(X_data, y_data)

y_pred = knn.predict(X_data)

print(accuracy(y_data, y_pred))

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, p=2)

knn = knn.fit(X_data, y_data)

y_pred = knn.predict(X_data)

print(accuracy(y_data, y_pred))

## Question 6

* Potriviți un model K-nearest neighbors folosind valori ale lui `k` (`n_neighbors`) cuprinse între 1 și 20. Utilizați ponderi uniforme (implicit). Coeficientul pentru distanța Minkowski (`p`) poate fi setat la 1 sau 2 - trebuie doar să fie consecvent. Stocați precizia și valoarea lui `k` utilizată pentru fiecare dintre aceste ajustări într-o listă sau într-un dicționar.
* Reprezentați grafic (sau vizualizați tabelul) `acuratețea` în funcție de `k`. Ce observați că se întâmplă atunci când `k=1`? De ce credeți că este așa? *Hint:* este din același motiv discutat mai sus.

In [ ]:
# Fit the K-nearest neighbors model with different values of k
# Store the accuracy measurement for each k

score_list = list()

for k in range(1, 21):
    
    knn = KNeighborsClassifier(n_neighbors=k)
    knn = knn.fit(X_data, y_data)
    
    y_pred = knn.predict(X_data)
    score = accuracy(y_data, y_pred)
    
    score_list.append((k, score))
    
score_df = pd.DataFrame(score_list, columns=['k', 'accuracy'])

In [ ]:
# Import libraries to make the plot

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
sns.set_context('talk')
sns.set_style('ticks')
sns.set_palette('dark')

ax = score_df.set_index('k').plot()

ax.set(xlabel='k', ylabel='accuracy')
ax.set_xticks(range(1, 21));